In [ ]:
!pip install -q sacrebleu openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/MWE/MWE GL-EN VBO-OBX - Hoja 1.csv'

In [ ]:
import pandas as pd
import sacrebleu
from pathlib import Path

# Ruta al archivo en Google Drive
file_path = Path("/content/drive/MyDrive/TFM/EN-GL MWEs annotation (1).xlsx")

# Cargar las dos hojas
sheets = {
    "EN→GL": pd.read_excel(file_path, sheet_name="EN-GL"),
    "GL→EN": pd.read_excel(file_path, sheet_name="GL-EN")
}

# Eliminar filas vacías arrastradas por formato
for direction, df in sheets.items():
    df = df.dropna(subset=["mwe_id", "sense_id"])
    sheets[direction] = df
    print(direction, df.shape)
    print(df.columns.tolist())
    print()

EN→GL (347, 18)
['mwe_id', 'sense_id', 'trans_man2_EN (TESTE)', 'manual_sent_2', 'trans_expr_EN', 'gold_mwe', 'googletranslate_output', 'google_trans_expr_GL', 'google_mwe_eval', 'nosmt_output', 'nosmt_trans_expr_GL', 'nosmt_mwe_eval', 'finetunednosmt_output', 'ftnosmt_trans_expr_GL', 'finetunednosmt_mwe_eval', 'salamandrata_output', 'salamandrata_trans_expr_GL', 'salamandrata_mwe_eval']

GL→EN (347, 19)
['mwe_id', 'sense_id', 'manual_sent_2', 'trans_man2_EN (TESTE)', 'mwe', 'gold_mwe', 'googletranslate_output', 'google_trans_expr_EN', 'google_mwe_eval', 'salamandrata_output', 'salamandrata_trans_expr_EN', 'salamandrata_mwe_eval', 'nosmt_output', 'nosmt_trans_expr_EN', 'nosmt_mwe_eval', 'finetuned_nosmt_output', 'ftnosmt_trans_expr_EN', 'finetunednosmt_mwe_eval', 'Unnamed: 18']



In [ ]:
# Columnas de referencia por dirección
# EN→GL: source = English sentence, reference = Galician sentence
# GL→EN: source = Galician sentence, reference = English sentence

reference_columns = {
    "EN→GL": "manual_sent_2",
    "GL→EN": "trans_man2_EN (TESTE)"
}

# Sistemas incluidos en la comparación principal
# Fine-tuned NósMT queda excluido
system_columns = {
    "Google": "googletranslate_output",
    "SalamandraTA": "salamandrata_output",
    "Nos_MT": "nosmt_output"
}

In [ ]:
def clean_text_series(series):
    """
    Converts a pandas Series to clean strings for metric calculation.
    It removes missing values, normalizes whitespace and strips spaces.
    """
    return (
        series
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .tolist()
    )


def compute_metrics(predictions, references):
    """
    Computes corpus-level BLEU, chrF and TER using sacreBLEU.
    references must be a single list of reference translations.
    """
    refs_for_sacrebleu = [references]

    bleu = sacrebleu.corpus_bleu(predictions, refs_for_sacrebleu)
    chrf = sacrebleu.corpus_chrf(predictions, refs_for_sacrebleu)
    ter = sacrebleu.corpus_ter(predictions, refs_for_sacrebleu)

    return {
        "BLEU": round(bleu.score, 2),
        "chrF": round(chrf.score, 2),
        "TER": round(ter.score, 2)
    }

In [ ]:
results = []

for direction, df in sheets.items():
    ref_col = reference_columns[direction]

    # Verificación básica
    if ref_col not in df.columns:
        raise ValueError(f"Reference column not found in {direction}: {ref_col}")

    for system_name, pred_col in system_columns.items():
        if pred_col not in df.columns:
            raise ValueError(f"Prediction column not found in {direction}: {pred_col}")

        # Mantener solo filas con referencia y output del sistema
        eval_df = df.dropna(subset=[ref_col, pred_col]).copy()

        references = clean_text_series(eval_df[ref_col])
        predictions = clean_text_series(eval_df[pred_col])

        metrics = compute_metrics(predictions, references)

        results.append({
            "Direction": direction,
            "System": system_name,
            "N": len(eval_df),
            "BLEU": metrics["BLEU"],
            "chrF": metrics["chrF"],
            "TER": metrics["TER"]
        })

results_df = pd.DataFrame(results)

display(results_df)

,Direction,System,N,BLEU,chrF,TER
0,EN→GL,Google,347,39.89,64.85,45.86
1,EN→GL,SalamandraTA,347,35.94,62.14,50.11
2,EN→GL,Nos_MT,347,32.40,59.66,52.86
3,GL→EN,Google,347,41.38,65.70,44.59
4,GL→EN,SalamandraTA,347,37.08,61.44,47.37
5,GL→EN,Nos_MT,347,34.87,60.79,50.44


In [ ]:
output_path = Path("/content/drive/MyDrive/TFM/automatic_evaluation_results.xlsx")

results_df.to_excel(output_path, index=False)

print(f"Results saved to: {output_path}")

Results saved to: /content/drive/MyDrive/TFM/automatic_evaluation_results.xlsx


In [ ]:
print("Direction\tSystem\tN\tBLEU\tchrF\tTER")

for _, row in results_df.iterrows():
    print(
        f"{row['Direction']}\t"
        f"{row['System']}\t"
        f"{row['N']}\t"
        f"{row['BLEU']}\t"
        f"{row['chrF']}\t"
        f"{row['TER']}"
    )

Direction	System	N	BLEU	chrF	TER
EN→GL	Google	347	39.89	64.85	45.86
EN→GL	SalamandraTA	347	35.94	62.14	50.11
EN→GL	Nos_MT	347	32.4	59.66	52.86
GL→EN	Google	347	41.38	65.7	44.59
GL→EN	SalamandraTA	347	37.08	61.44	47.37
GL→EN	Nos_MT	347	34.87	60.79	50.44
